In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer

In [23]:
df_10k = pd.read_csv("/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/1_sources/Journal Articles_10k_Titles_Authors_Year_Topic_Publisher.csv", header=0, index_col=0)
df_10k
df_20k = pd.read_csv("/Users/huss/data-processing/Github/Feminist-Literature-Engine-ML-Web-Scraping-App-/1_sources/Journal Articles_20k_Titles_Authors_Year_Topic_Publisher.csv", header=0, index_col=0)
df_20k
df = pd.concat([df_10k, df_20k], axis=0)
df

,Title,Year,Citations,Authors,AuthorOrder,Venue,AbstractInvertedIndex,AbstractText,Publisher,OpenAccess,...,Topics,Domain,Field,Subfield,Institutions,Languages,WorkType,DOI,LandingPage,PDFURL
ID,,,,,,,,,,,,,,,,,,,,,
1,Gender Trouble: Feminism and the Subversion of...,1991.0,28049,Mary McIntosh; Judith Butler,1. Mary McIntosh; 2. Judith Butler,Feminist Review,"{""Preface"": [0, 2], ""(1999)"": [1], ""(1990)"": [...",Preface (1999) Preface (1990) 1. Subjects of S...,SAGE Publishing,False,...,Latin American and Latino Studies; Anarchism a...,Social Sciences,Social Sciences,Cultural Studies,NaN,en,article,https://doi.org/10.2307/1395391,https://doi.org/10.2307/1395391,NaN
2,Situated Knowledges: The Science Question in F...,1988.0,17241,Donna Haraway,1. Donna Haraway,Feminist Studies,NaN,NaN,Feminist Studies,False,...,Feminist Epistemology and Gender Studies; Cont...,Social Sciences,Social Sciences,Sociology and Political Science,NaN,en,article,https://doi.org/10.2307/3178066,https://doi.org/10.2307/3178066,NaN
3,Gender trouble: feminism and the subversion of...,1990.0,7755,NaN,NaN,Choice Reviews Online,NaN,NaN,Association of College and Research Libraries,False,...,Gender Politics and Representation; Historical...,Social Sciences,Social Sciences,Gender Studies,NaN,en,article,https://doi.org/10.5860/choice.28-1264,https://doi.org/10.5860/choice.28-1264,NaN
4,Situated Knowledges: The Science Question in F...,1988.0,7010,Donna Haraway,1. Donna Haraway,PhilPapers (PhilPapers Foundation),"{""Academic"": [0], ""and"": [1, 23, 34, 44, 60, 6...",Academic and activist feminist inquiry has rep...,NaN,True,...,Interdisciplinary Research and Collaboration; ...,Social Sciences,Decision Sciences,Information Systems and Management,NaN,en,article,NaN,https://philarchive.org/rec/HARSKT,https://philpapers.org/archive/HARSKT.pdf
5,The science question in feminism,1987.0,3829,Kristin Waters,1. Kristin Waters,Women s Studies International Forum,NaN,NaN,Elsevier BV,False,...,Species Distribution and Climate Change; Susta...,Physical Sciences,Environmental Science,Ecological Modeling,NaN,en,article,https://doi.org/10.1016/0277-5395(87)90077-x,https://doi.org/10.1016/0277-5395(87)90077-x,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,Gender matters: Feminist research in education...,2002.0,18,Wanda S. Pillow,1. Wanda S. Pillow,New Directions for Evaluation,"{""Abstract"": [0], ""The"": [1], ""key"": [2], ""ins...",Abstract The key insights and shifts in first‐...,Wiley,False,...,Student Assessment and Feedback; Evaluation an...,Social Sciences,Social Sciences,Education,University of Illinois Urbana-Champaign,en,article,https://doi.org/10.1002/ev.63,https://doi.org/10.1002/ev.63,NaN
9997,Subverting and Minding Boundaries: The Intelle...,2018.0,75,Leslie D. Gonzales,1. Leslie D. Gonzales,The Journal of Higher Education,"{""Using"": [0], ""various"": [1], ""methods"": [2],...","Using various methods and analytical angles, r...",Taylor & Francis,False,...,Gender Diversity and Inequality,Social Sciences,Social Sciences,Gender Studies,Michigan State University,en,article,https://doi.org/10.1080/00221546.2018.1434278,https://doi.org/10.1080/00221546.2018.1434278,NaN
9998,Gender and History,2011.0,34,Susan Kingsley Kent,1. Susan Kingsley Kent,Medical Entomology and Zoology,"{""Acknowledgements.-"": [0], ""Introduction:"": [...","Acknowledgements.- Introduction: History, Theo...",Japan Society of Medical Entomology and Zoology,False,...,Historical Gender and Feminism Studies,Social Sciences,Social Sciences,Sociology and Political Science,NaN,en,book,NaN,http://ci.nii.ac.jp/ncid/BB07753143,NaN


In [24]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace('(', '_', regex=False)   # replace ( with _
    .str.replace('/', '_', regex=False)   # replace ( with _
    .str.replace(')', '', regex=False)    # remove )
    .str.replace(r'[^\w\s]', '', regex=True)  # remove other special chars
    .str.replace(' ', '_')                # spaces → underscore
    .str.replace('__', '_')
)
df.columns

Index(['title', 'year', 'citations', 'authors', 'authororder', 'venue',
       'abstractinvertedindex', 'abstracttext', 'publisher', 'openaccess',
       'oastatus', 'referencedworks', 'relatedworks', 'concepts', 'topics',
       'domain', 'field', 'subfield', 'institutions', 'languages', 'worktype',
       'doi', 'landingpage', 'pdfurl'],
      dtype='str')

In [25]:
import numpy as np
#Explore all data
#print("\n", "Head:","\n", df.head(5))
#print("\n", "Tail:","\n", df.tail(5))
print("\n", "Null Values:","\n", df.isna().sum())




 Null Values: 
 title                        0
year                        13
citations                    0
authors                   1200
authororder               1200
venue                     3101
abstractinvertedindex     6127
abstracttext              6145
publisher                 6074
openaccess                   0
oastatus                     0
referencedworks          11340
relatedworks               740
concepts                     4
topics                      35
domain                      35
field                       35
subfield                    35
institutions             12298
languages                   10
worktype                     0
doi                       3753
landingpage                516
pdfurl                   17432
dtype: int64


In [27]:
df = df.drop(["pdfurl", "venue", "authororder", "", "abstractinvertedindex", "abstracttext", "publisher", "referencedworks", "institutions", "doi", "landingpage"], axis=1)
df

KeyError: "['pdfurl', 'venue', 'authororder', 'abstractinvertedindex', 'abstracttext', 'publisher', 'referencedworks', 'institutions', 'doi', 'landingpage'] not found in axis"

In [29]:
df = df.drop(["oastatus"], axis=1)
df

,title,year,citations,authors,openaccess,relatedworks,concepts,topics,domain,field,subfield,languages,worktype
ID,,,,,,,,,,,,,
1,Gender Trouble: Feminism and the Subversion of...,1991.0,28049,Mary McIntosh; Judith Butler,False,https://openalex.org/W2982445252; https://open...,Subversion; Feminism; Gender studies; Sociolog...,Latin American and Latino Studies; Anarchism a...,Social Sciences,Social Sciences,Cultural Studies,en,article
2,Situated Knowledges: The Science Question in F...,1988.0,17241,Donna Haraway,False,https://openalex.org/W1946080426; https://open...,Situated; Perspective (graphical); Privilege (...,Feminist Epistemology and Gender Studies; Cont...,Social Sciences,Social Sciences,Sociology and Political Science,en,article
3,Gender trouble: feminism and the subversion of...,1990.0,7755,NaN,False,https://openalex.org/W2038600245; https://open...,Subversion; Feminism; Identity (music); Gender...,Gender Politics and Representation; Historical...,Social Sciences,Social Sciences,Gender Studies,en,article
4,Situated Knowledges: The Science Question in F...,1988.0,7010,Donna Haraway,True,https://openalex.org/W2753533763; https://open...,Situated; Epistemology; Sociology; Relativism;...,Interdisciplinary Research and Collaboration; ...,Social Sciences,Decision Sciences,Information Systems and Management,en,article
5,The science question in feminism,1987.0,3829,Kristin Waters,False,https://openalex.org/W3114154697; https://open...,Bridge (graph theory); Sustainability; Knowled...,Species Distribution and Climate Change; Susta...,Physical Sciences,Environmental Science,Ecological Modeling,en,article
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,Gender matters: Feminist research in education...,2002.0,18,Wanda S. Pillow,False,https://openalex.org/W4391375266; https://open...,Research methodology; Educational research; So...,Student Assessment and Feedback; Evaluation an...,Social Sciences,Social Sciences,Education,en,article
9997,Subverting and Minding Boundaries: The Intelle...,2018.0,75,Leslie D. Gonzales,False,https://openalex.org/W4391375266; https://open...,Work (physics); Gender studies; Boundary-work;...,Gender Diversity and Inequality,Social Sciences,Social Sciences,Gender Studies,en,article
9998,Gender and History,2011.0,34,Susan Kingsley Kent,False,https://openalex.org/W3135220427; https://open...,Glossary; Feminism; Gender studies; Reading (p...,Historical Gender and Feminism Studies,Social Sciences,Social Sciences,Sociology and Political Science,en,book


In [31]:
print("\n", "Null Values:","\n", df.isna().sum())



 Null Values: 
 title              0
year              13
citations          0
authors         1200
openaccess         0
relatedworks     740
concepts           4
topics            35
domain            35
field             35
subfield          35
languages         10
worktype           0
dtype: int64


In [32]:
df = df.dropna(how="any")
df
print("\n", "Null Values:","\n", df.isna().sum())


 Null Values: 
 title           0
year            0
citations       0
authors         0
openaccess      0
relatedworks    0
concepts        0
topics          0
domain          0
field           0
subfield        0
languages       0
worktype        0
dtype: int64


In [33]:
print("\n", "Info:","\n", df.info())
#print("\n", "Info Memory Usage:","\n", df.info(memory_usage="deep"))


<class 'pandas.DataFrame'>
Index: 18026 entries, 1 to 10000
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         18026 non-null  str    
 1   year          18026 non-null  float64
 2   citations     18026 non-null  int64  
 3   authors       18026 non-null  str    
 4   openaccess    18026 non-null  bool   
 5   relatedworks  18026 non-null  str    
 6   concepts      18026 non-null  str    
 7   topics        18026 non-null  str    
 8   domain        18026 non-null  str    
 9   field         18026 non-null  str    
 10  subfield      18026 non-null  str    
 11  languages     18026 non-null  str    
 12  worktype      18026 non-null  str    
dtypes: bool(1), float64(1), int64(1), str(10)
memory usage: 15.3 MB

 Info: 
 None


In [34]:
print("\n", "Describe:","\n", df.describe())
print("\n", "Describe All:","\n", df.describe(include="all"))
print("\n", "Shape:","\n", df.shape)
print("\n", "DTypes:","\n", df.dtypes)
print("\n", "Columns:","\n", df.columns)
#print("\n", "Index:","\n", df.index)
print("\n", "Values:","\n", df.values)
print("\n", "Empty:","\n", df.empty)
#print("\n", "NDim:","\n", df.ndim)
print("\n", "Size:","\n", df.size)
#print("\n", "Memory True:","\n", df.memory_usage(deep=True))
#print("\n", "Transpose:","\n", df.T)


 Describe: 
                year     citations
count  18026.000000  18026.000000
mean    2007.338456     58.341174
std       12.453088    431.409514
min     1814.000000      1.000000
25%     1999.000000      2.000000
50%     2010.000000      8.000000
75%     2018.000000     39.000000
max     2026.000000  28049.000000

 Describe All: 
            title          year     citations          authors openaccess  \
count      18026  18026.000000  18026.000000            18026      18026   
unique     17108           NaN           NaN            13795          2   
top     Feminism           NaN           NaN  Ángela McRobbie      False   
freq          34           NaN           NaN               26      14214   
mean         NaN   2007.338456     58.341174              NaN        NaN   
std          NaN     12.453088    431.409514              NaN        NaN   
min          NaN   1814.000000      1.000000              NaN        NaN   
25%          NaN   1999.000000      2.000000          

In [35]:
#split cols into different columns 
numerical_cols = df.select_dtypes(include=np.number)
print(numerical_cols)

#be careful because some categorical values my sip into numerical_cols (numbers that represent category)
categorical_cols = df.select_dtypes(exclude=np.number)
print(categorical_cols)

         year  citations
ID                      
1      1991.0      28049
2      1988.0      17241
4      1988.0       7010
5      1987.0       3829
6      1995.0       5048
...       ...        ...
9996   2002.0         18
9997   2018.0         75
9998   2011.0         34
9999   2017.0         22
10000  2016.0         38

[18026 rows x 2 columns]
                                                   title  \
ID                                                         
1      Gender Trouble: Feminism and the Subversion of...   
2      Situated Knowledges: The Science Question in F...   
4      Situated Knowledges: The Science Question in F...   
5                       The science question in feminism   
6      Unbearable Weight: Feminism, Western Culture a...   
...                                                  ...   
9996   Gender matters: Feminist research in education...   
9997   Subverting and Minding Boundaries: The Intelle...   
9998                                  Gender and 

In [36]:
df["year"] = df["year"].astype(int)
df.head()

,title,year,citations,authors,openaccess,relatedworks,concepts,topics,domain,field,subfield,languages,worktype
ID,,,,,,,,,,,,,
1,Gender Trouble: Feminism and the Subversion of...,1991,28049,Mary McIntosh; Judith Butler,False,https://openalex.org/W2982445252; https://open...,Subversion; Feminism; Gender studies; Sociolog...,Latin American and Latino Studies; Anarchism a...,Social Sciences,Social Sciences,Cultural Studies,en,article
2,Situated Knowledges: The Science Question in F...,1988,17241,Donna Haraway,False,https://openalex.org/W1946080426; https://open...,Situated; Perspective (graphical); Privilege (...,Feminist Epistemology and Gender Studies; Cont...,Social Sciences,Social Sciences,Sociology and Political Science,en,article
4,Situated Knowledges: The Science Question in F...,1988,7010,Donna Haraway,True,https://openalex.org/W2753533763; https://open...,Situated; Epistemology; Sociology; Relativism;...,Interdisciplinary Research and Collaboration; ...,Social Sciences,Decision Sciences,Information Systems and Management,en,article
5,The science question in feminism,1987,3829,Kristin Waters,False,https://openalex.org/W3114154697; https://open...,Bridge (graph theory); Sustainability; Knowled...,Species Distribution and Climate Change; Susta...,Physical Sciences,Environmental Science,Ecological Modeling,en,article
6,"Unbearable Weight: Feminism, Western Culture a...",1995,5048,Mimi Nichter; Susan Bordo,False,https://openalex.org/W2748952813; https://open...,Feminism; Gender studies; Sociology; Political...,French Historical and Cultural Studies,Social Sciences,Arts and Humanities,History,en,article


In [37]:
print("Shape:", df.shape)
print("\nColumn names:\n", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nFirst 3 rows:")
df.head(3)

Shape: (18026, 13)

Column names:
 ['title', 'year', 'citations', 'authors', 'openaccess', 'relatedworks', 'concepts', 'topics', 'domain', 'field', 'subfield', 'languages', 'worktype']

Data types:
 title             str
year            int64
citations       int64
authors           str
openaccess       bool
relatedworks      str
concepts          str
topics            str
domain            str
field             str
subfield          str
languages         str
worktype          str
dtype: object

First 3 rows:


,title,year,citations,authors,openaccess,relatedworks,concepts,topics,domain,field,subfield,languages,worktype
ID,,,,,,,,,,,,,
1,Gender Trouble: Feminism and the Subversion of...,1991,28049,Mary McIntosh; Judith Butler,False,https://openalex.org/W2982445252; https://open...,Subversion; Feminism; Gender studies; Sociolog...,Latin American and Latino Studies; Anarchism a...,Social Sciences,Social Sciences,Cultural Studies,en,article
2,Situated Knowledges: The Science Question in F...,1988,17241,Donna Haraway,False,https://openalex.org/W1946080426; https://open...,Situated; Perspective (graphical); Privilege (...,Feminist Epistemology and Gender Studies; Cont...,Social Sciences,Social Sciences,Sociology and Political Science,en,article
4,Situated Knowledges: The Science Question in F...,1988,7010,Donna Haraway,True,https://openalex.org/W2753533763; https://open...,Situated; Epistemology; Sociology; Relativism;...,Interdisciplinary Research and Collaboration; ...,Social Sciences,Decision Sciences,Information Systems and Management,en,article


In [52]:
#Count Unique values 
print("\n", df["year"].value_counts().sum)
print("\n", df["openaccess"].value_counts().sum)
print("\n", df["languages"].value_counts().sum)
print("\n", df["worktype"].value_counts().sum)
print("\n", df["domain"].value_counts().sum)
print("\n", df["field"].value_counts().sum)


 <bound method Series.sum of year
2019    798
2018    786
2020    767
2017    740
2016    696
       ... 
1930      1
1962      1
1957      1
1928      1
1937      1
Name: count, Length: 91, dtype: int64>

 <bound method Series.sum of openaccess
False    14214
True      3812
Name: count, dtype: int64>

 <bound method Series.sum of languages
en     17749
fr        79
es        64
de        30
id        18
pt        17
it        14
sv         8
ko         7
tr         6
fi         4
ja         4
nl         4
mk         3
ar         3
ru         2
ceb        2
ca         2
pl         2
sk         1
cs         1
cy         1
no         1
pms        1
ku         1
ms         1
uk         1
Name: count, dtype: int64>

 <bound method Series.sum of worktype
article            12373
book-chapter        2566
book                2492
other                202
dissertation         194
review                87
preprint              41
editorial             23
reference-entry       18
dataset       

In [ ]:
#### TITLE ANALYSIS

In [60]:
from sklearn.feature_extraction.text import CountVectorizer

titles = df['title'].dropna().astype(str)

vectorizer = CountVectorizer(
    stop_words='english',     # removes common words automatically
    ngram_range=(1, 2),       # counts single words AND two-word phrases
    max_features=500           # top 50 most frequent terms
)

X = vectorizer.fit_transform(titles)

# Sum occurrences of each word across all titles
word_counts = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'count': X.toarray().sum(axis=0)
}).sort_values('count', ascending=False)

print(word_counts.head(30))

              word  count
135       feminism  14223
484          women   2727
163       feminist   1413
162      feminisms   1325
177         gender   1281
325       politics    940
404         social    661
443         theory    651
302            new    636
30           black    526
82        critical    417
483          woman    385
324      political    380
473           wave    379
285       movement    372
200        history    365
429        studies    361
77    contemporary    352
453  transnational    350
332           post    347
88         culture    345
12        american    345
113      education    324
18            anti    311
421          state    309
188         global    295
387         rights    294
271          media    291
377       research    286
16        analysis    281


In [ ]:
#### CONCEPTS ANALYSIS

In [62]:

titles = df['concepts'].dropna().astype(str)

vectorizer = CountVectorizer(
    stop_words='english',     # removes common words automatically
    ngram_range=(1, 2),       # counts single words AND two-word phrases
    max_features=500           # top 50 most frequent terms
)

X = vectorizer.fit_transform(titles)

# Sum occurrences of each word across all titles
word_counts = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'count': X.toarray().sum(axis=0)
}).sort_values('count', ascending=False)

print(word_counts.head(30))

                   word  count
450             studies  18644
386             science  17583
424           sociology  17414
153              gender  16759
155      gender studies  16268
128            feminism  16037
313           political  12092
315   political science  11138
204                 law  10090
316            politics   7518
291          philosophy   7463
26                  art   6097
344          psychology   5201
170             history   4544
417              social   4079
67             computer   4020
469   studies sociology   4009
68     computer science   3802
431    sociology gender   3318
400         science law   3225
111        epistemology   3081
306             physics   2945
21          archaeology   2681
132     feminism gender   2476
4            aesthetics   2298
46              biology   2045
234          literature   1998
231         linguistics   1951
144  feminism sociology   1900
476              theory   1897


In [ ]:
#### TOPICS ANALYSIS

In [70]:

titles = df['topics'].dropna().astype(str)

vectorizer = CountVectorizer(
    stop_words='english',     # removes common words automatically
    ngram_range=(1, 2),       # counts single words AND two-word phrases
    max_features=500           # top 50 most frequent terms
)

X = vectorizer.fit_transform(titles)

# Sum occurrences of each word across all titles
word_counts = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'count': X.toarray().sum(axis=0)
}).sort_values('count', ascending=False)

print(word_counts.head(30))

                        word  count
159                   gender  12541
441                  studies  10914
144                 feminism   5272
163          gender feminism   4680
328                 politics   4285
275                    media   4246
189               historical   4205
196                  history   3944
192        historical gender   2359
147         feminism studies   2359
372           representation   2351
146           feminism media   2320
334  politics representation   2279
166          gender politics   2279
417                  society   2221
119                education   2159
405                   social   1851
10                  american   1815
207                 identity   1787
322                political   1781
87                   culture   1717
182                   health   1568
80                  cultural   1539
131                   ethics   1469
264               literature   1410
103                 dynamics   1310
93               development

In [ ]:
#### DOMAIN ANALYSIS

In [64]:

titles = df['domain'].dropna().astype(str)

vectorizer = CountVectorizer(
    stop_words='english',     # removes common words automatically
    ngram_range=(1, 2),       # counts single words AND two-word phrases
    max_features=500           # top 50 most frequent terms
)

X = vectorizer.fit_transform(titles)

# Sum occurrences of each word across all titles
word_counts = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'count': X.toarray().sum(axis=0)
}).sort_values('count', ascending=False)

print(word_counts.head(30))

                word  count
6           sciences  18026
7             social  17157
8    social sciences  17157
0             health    504
1    health sciences    504
4           physical    218
5  physical sciences    218
2               life    147
3      life sciences    147


In [ ]:
#### FIELD ANALYSIS

In [67]:

titles = df['field'].dropna().astype(str)

vectorizer = CountVectorizer(
    stop_words='english',     # removes common words automatically
    ngram_range=(1, 2),       # counts single words AND two-word phrases
    max_features=500           # top 50 most frequent terms
)

X = vectorizer.fit_transform(titles)

# Sum occurrences of each word across all titles
word_counts = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'count': X.toarray().sum(axis=0)
}).sort_values('count', ascending=False)

print(word_counts.head(30))

                       word  count
54                 sciences  13092
56          social sciences  12957
55                   social  12957
33               humanities   2996
3                      arts   2996
4           arts humanities   2996
52               psychology    805
39                 medicine    261
20             econometrics    242
21     econometrics finance    242
22                economics    242
23   economics econometrics    242
28                  finance    242
31                   health    213
51              professions    213
32       health professions    213
53                  science    186
0                accounting    121
34               management    121
35    management accounting    121
12      business management    121
11                 business    121
26            environmental    114
27    environmental science    114
1              agricultural     96
9       biological sciences     96
8                biological     96
2   agricultural bio

In [ ]:
#### SUBFIELD ANALYSIS

In [66]:

titles = df['subfield'].dropna().astype(str)

vectorizer = CountVectorizer(
    stop_words='english',     # removes common words automatically
    ngram_range=(1, 2),       # counts single words AND two-word phrases
    max_features=500           # top 50 most frequent terms
)

X = vectorizer.fit_transform(titles)

# Sum occurrences of each word across all titles
word_counts = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'count': X.toarray().sum(axis=0)
}).sort_values('count', ascending=False)

print(word_counts.head(30))

                        word  count
267                  science   6477
242        political science   6345
241                political   6345
291                  studies   5054
277                sociology   5001
278      sociology political   5001
105                   gender   4227
106           gender studies   4227
142  international relations   1344
141            international   1344
254                relations   1344
269    science international   1344
248               psychology    805
14                      arts    753
125                  history    742
226               philosophy    696
300                   theory    681
163               literature    672
164      literature literary    672
161                 literary    672
162          literary theory    672
60                  cultural    578
61          cultural studies    578
119                   health    429
76                 education    394
220               performing    375
16           arts performing